# Reinforcement Learning

Reinforcement Learning (RL) is a type of machine learning where an **agent learns by interacting with an environment** and receiving rewards or penalties.

## Basic Workflow

**Environment → State → Agent → Action → Reward → Next State → Learn/Update Policy → Repeat**

Unlike Regression and Classification, RL does not normally use a fixed `X` and `y` dataset.

## Core Components

- **Agent** → learner/decision maker
- **Environment** → world in which the agent acts
- **State** → current situation
- **Action** → decision made by the agent
- **Reward** → feedback from the environment
- **Policy** → strategy for choosing actions
- **Value Function** → expected future reward
- **Q-Value** → expected reward for taking an action in a state
- **Episode** → one complete interaction from start to terminal state

## Exploration vs Exploitation

**Exploration** → try new actions to discover better strategies

**Exploitation** → choose the action currently believed to be best

A common strategy is **epsilon-greedy**:

```text
With probability ε → explore
With probability 1-ε → exploit
```

Libraries / Tools

In [ ]:
# Common RL tools

# pip install gymnasium stable-baselines3 torch

import numpy as np

# Gymnasium
# → environments and RL interfaces

# Stable-Baselines3
# → ready-to-use RL algorithms

# PyTorch / TensorFlow
# → neural-network implementations for deep RL

# Simple RL loop structure

In [ ]:
for episode in range(100):
    state = env.reset()

    done = False

    while not done:
        action = agent.choose_action(state)

        next_state, reward, done, info = env.step(action)

        agent.learn(
            state,
            action,
            reward,
            next_state
        )

        state = next_state

# Environment

**Use:** Define the states, actions, rewards and transition rules.

Gymnasium provides standard environments for learning and experimentation.

In [ ]:
# Example Gymnasium environment

import gymnasium as gym

env = gym.make("CartPole-v1")

observation, info = env.reset(seed=42)

print("Initial state:", observation)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)

# Random Policy Baseline

**Use:** Establish a very simple baseline before training an RL algorithm.

In [ ]:
episode_rewards = []

for episode in range(5):
    observation, info = env.reset()
    total_reward = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = env.action_space.sample()

        observation, reward, terminated, truncated, info = env.step(action)

        total_reward += reward

    episode_rewards.append(total_reward)

print("Random policy rewards:", episode_rewards)
print("Average reward:", np.mean(episode_rewards))

# Q-Learning

**Type:** Value-based, model-free RL

**Use:** Discrete state/action problems where you can represent a Q-table.

Main idea:

**Learn Q(state, action) → choose actions with high expected future reward**

### Q-Learning Update

```text
Q(s,a) ← Q(s,a) + α [r + γ max Q(s',a') - Q(s,a)]
```

- `α` → learning rate
- `γ` → discount factor
- `r` → reward
- `s'` → next state

In [ ]:
# Simple Q-learning skeleton for a discrete environment

from collections import defaultdict

Q = defaultdict(lambda: defaultdict(float))

alpha = 0.1
gamma = 0.99
epsilon = 1.0

def choose_action(state, action_space_n):
    if np.random.rand() < epsilon:
        return np.random.randint(action_space_n)

    values = [
        Q[state][a]
        for a in range(action_space_n)
    ]

    return int(np.argmax(values))

# Q-learning is easiest to demonstrate
# with a small discrete-state environment.
# For large/continuous state spaces,
# use function approximation / deep RL.

# SARSA

**Type:** Value-based, model-free RL

**Use:** Similar to Q-learning, but learns from the action actually selected by the current policy.

Main difference:

**Q-Learning → off-policy**

**SARSA → on-policy**

In [ ]:
# SARSA update

# Q(s,a) ← Q(s,a) +
# α [r + γ Q(s',a') - Q(s,a)]
#
# The next action a' is the action actually chosen
# by the current policy.

# Deep Q-Network (DQN)

**Type:** Deep value-based RL

**Use:** Large/complex state spaces where a neural network can approximate Q-values instead of storing a Q-table.

Core ideas:
- Neural network
- Experience replay
- Target network
- Epsilon-greedy exploration

In [ ]:
# Stable-Baselines3 DQN example

from stable_baselines3 import DQN

env = gym.make("CartPole-v1")

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.1,
    verbose=0,
    seed=42
)

model.learn(total_timesteps=10000)

# Policy Gradient

**Type:** Policy-based RL

**Use:** Learn a policy directly instead of learning only a Q-table.

The model learns:

**State → Probability distribution over actions**

# Actor-Critic

**Type:** Combines policy-based and value-based learning.

- **Actor** → chooses actions / learns policy
- **Critic** → evaluates actions using value estimates

This idea is used by many modern RL algorithms.

# PPO (Proximal Policy Optimization)

**Type:** Policy-gradient / actor-critic family

**Use:** A popular general-purpose deep RL algorithm that constrains policy updates to improve training stability.

PPO is often a practical starting point for modern continuous/discrete-control experiments.

In [ ]:
from stable_baselines3 import PPO

env = gym.make("CartPole-v1")

model_ppo = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    verbose=0,
    seed=42
)

model_ppo.learn(total_timesteps=10000)

# A2C (Advantage Actor-Critic)

**Type:** Actor-Critic

**Use:** Learns a policy and value function together using an advantage estimate.

Advantage roughly asks:

**Was this action better or worse than expected?**

In [ ]:
from stable_baselines3 import A2C

env = gym.make("CartPole-v1")

model_a2c = A2C(
    "MlpPolicy",
    env,
    learning_rate=7e-4,
    gamma=0.99,
    verbose=0,
    seed=42
)

model_a2c.learn(total_timesteps=10000)

# DDPG

**Type:** Actor-Critic, off-policy

**Use:** Continuous action spaces.

It learns:
- Actor → continuous actions
- Critic → Q-value of state/action

DDPG is useful for understanding continuous-control actor-critic methods.

In [ ]:
from stable_baselines3 import DDPG

# DDPG requires a continuous-action environment.
# Example:
#
# env = gym.make("Pendulum-v1")
#
# model_ddpg = DDPG(
#     "MlpPolicy",
#     env,
#     verbose=0,
#     seed=42
# )
#
# model_ddpg.learn(total_timesteps=10000)

# SAC (Soft Actor-Critic)

**Type:** Off-policy actor-critic

**Use:** Continuous control with an entropy-based objective that encourages useful exploration.

SAC is commonly used for continuous-action problems.

In [ ]:
from stable_baselines3 import SAC

# Example for continuous actions:
#
# env = gym.make("Pendulum-v1")
#
# model_sac = SAC(
#     "MlpPolicy",
#     env,
#     verbose=0,
#     seed=42
# )
#
# model_sac.learn(total_timesteps=10000)

# TD3 (Twin Delayed DDPG)

**Type:** Off-policy actor-critic

**Use:** Continuous-control problems where reducing overestimation and stabilizing DDPG-style learning is useful.

In [ ]:
from stable_baselines3 import TD3

# Example:
#
# env = gym.make("Pendulum-v1")
#
# model_td3 = TD3(
#     "MlpPolicy",
#     env,
#     verbose=0,
#     seed=42
# )
#
# model_td3.learn(total_timesteps=10000)

# Training Loop

The general RL loop is:

```text
Reset environment
      ↓
Observe state
      ↓
Choose action
      ↓
Take action
      ↓
Receive reward
      ↓
Observe next state
      ↓
Update policy/value
      ↓
Repeat
```

This repeats for many steps and episodes.

In [ ]:
# Generic interaction loop after training

obs, info = env.reset(seed=42)

total_reward = 0
terminated = False
truncated = False

while not (terminated or truncated):
    action, _states = model_ppo.predict(
        obs,
        deterministic=True
    )

    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

print("Episode reward:", total_reward)

# Evaluation

RL is usually evaluated over **multiple episodes**, not one prediction.

Common metrics:
- **Mean episode reward** → higher is usually better
- **Reward standard deviation** → consistency
- **Episode length** → useful depending on environment
- **Success rate** → percentage of successful episodes
- **Average return**
- **Training stability**

Always use evaluation episodes separate from training exploration when possible.

In [ ]:
def evaluate_agent(model, env, n_episodes=10):
    rewards = []

    for _ in range(n_episodes):
        obs, info = env.reset()

        total_reward = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            action, _ = model.predict(
                obs,
                deterministic=True
            )

            obs, reward, terminated, truncated, info = env.step(action)

            total_reward += reward

        rewards.append(total_reward)

    return {
        "Mean Reward": np.mean(rewards),
        "Std Reward": np.std(rewards),
        "Min Reward": np.min(rewards),
        "Max Reward": np.max(rewards)
    }

evaluate_agent(model_ppo, env, n_episodes=10)

# Model Comparison

Compare algorithms using the **same environment, reward definition, evaluation protocol and comparable training budget**.

| Algorithm | Main idea | Best use |
|---|---|---|
| Q-Learning | Q-table, off-policy | Small discrete problems |
| SARSA | On-policy Q-learning | Discrete problems |
| DQN | Neural network estimates Q-values | Large/discrete state spaces |
| Policy Gradient | Learn policy directly | Complex policies |
| Actor-Critic | Actor + value critic | Larger/continuous control |
| PPO | Stable clipped policy updates | General modern RL/control |

### Quick selection

**Small discrete problem → Q-Learning / SARSA**

**Large discrete state space → DQN**

**Complex or continuous control → Actor-Critic / PPO**

Compare:
- Mean reward
- Reward stability
- Success rate
- Training time
- Sample efficiency

Do not select an RL model just because it has the highest reward from one lucky episode.

In [ ]:
# Example comparison table

comparison = pd.DataFrame([
    {"Model": "DQN", "Mean Reward": np.nan, "Std Reward": np.nan},
    {"Model": "PPO", "Mean Reward": np.nan, "Std Reward": np.nan},
    {"Model": "A2C", "Mean Reward": np.nan, "Std Reward": np.nan},
])

comparison

# Model Improvement

**Baseline → Better Reward Design → Better Exploration → Hyperparameter Tuning → More Training → Evaluate Across Episodes → Final Policy**

### Reward Engineering
Design rewards that encourage the desired behavior.

Avoid rewards that can be exploited by the agent.

### Exploration
Tune:
- `epsilon` for epsilon-greedy methods
- Entropy/exploration settings for policy methods

### Hyperparameters
Common parameters:
- Learning rate
- Discount factor `gamma`
- Batch size
- Network architecture
- Number of steps
- Replay buffer size
- Exploration parameters

### Experience Replay
Used by algorithms such as DQN/DDPG/SAC/TD3 to reuse past experiences.

### Target Networks
Used in value-based deep RL to improve training stability.

### Reward Scaling / Normalization
Can improve optimization when reward magnitudes are difficult for the algorithm.

### Observation Normalization
Can help when observations have very different scales.

### Evaluation
Use multiple episodes and preferably multiple random seeds because RL training can be highly variable.

In [ ]:
# Example hyperparameter search idea

learning_rates = [1e-4, 3e-4, 1e-3]
gammas = [0.95, 0.99]

# In RL, this is usually an experiment loop rather than
# blindly applying GridSearchCV.

experiments = []

for lr in learning_rates:
    for gamma in gammas:
        experiments.append({
            "learning_rate": lr,
            "gamma": gamma
        })

pd.DataFrame(experiments)

# Quick Revision

**Q-Learning** → value-based, off-policy, Q-table

**SARSA** → value-based, on-policy

**DQN** → neural-network Q-learning

**Policy Gradient** → directly learn policy

**Actor-Critic** → actor chooses, critic evaluates

**PPO** → stable/popular policy optimization

**A2C** → advantage actor-critic

**DDPG** → continuous actions, actor-critic

**SAC** → continuous control + entropy/exploration

**TD3** → improved DDPG-style continuous control

### Main workflow

**Environment → State → Action → Reward → Next State → Update → Repeat**

### Main improvement methods

**Reward Design → Exploration → Hyperparameter Tuning → Network/Algorithm Choice → Experience Replay → Normalization → More Training → Multi-Episode Evaluation**

### Main difference from supervised learning

Regression/Classification:

**X + y → train → predict → target-based metric**

Reinforcement Learning:

**Agent + Environment → actions → rewards → learn policy → evaluate return**